# Imports and preprocessing

In [79]:
import csv
import numpy as np
from pathlib import Path
import sys, os
import json
import importlib
sys.path.append(os.path.join(os.getcwd(), '..'))

from utils import eda_helpers, all_functions, cv, sampling_methods
import implementations

importlib.reload(eda_helpers)
importlib.reload(all_functions)
importlib.reload(cv)
importlib.reload(sampling_methods)
importlib.reload(implementations)

from utils.eda_helpers import clean_data, read_csv_numeric
from utils.all_functions import sigmoid
from implementations import (
    logistic_regression,
    reg_logistic_regression,
    weighted_logistic_regression,
)

file_path = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
DATA_FOLDER = os.path.join(file_path, '..', 'dataset')

In [2]:
x_indices, x_data = read_csv_numeric(os.path.join(DATA_FOLDER, "x_train.csv"))
y_indices, y_data = read_csv_numeric(os.path.join(DATA_FOLDER, "y_train.csv"))

In [80]:
final_feature_set = ['_CASTHM1', 'FLUSHOT6', 'METVL11_', '_PASTRNG', 'EDUCA', 'CHCCOPD1', 'DRADVISE', 
                     'CHCKIDNY', 'ARTHSOCL', 'MENTHLTH', 'SMOKE100', 'DOCTDIAB', 'DIABAGE2', 'VETERAN3', 
                     'CHKHEMO3', 'SEX', '_DRNKWEK', 'JOINPAIN', 'MARITAL', 'EXERANY2', '_PACAT1', 'INSULIN', 
                     'DIFFALON', 'FRUIT1', 'QLACTLM2', 'BLIND', 'PADUR2_', 'PAFREQ2_', 'LMTJOIN3', 'CHOLCHK', 
                     'POORHLTH', 'DIABETE3', '_FRUTSUM', 'PERSDOC2', 'EXRACT21', 'FEETCHK', '_VEGESUM', 'PNEUVAC3', 
                     'BPHIGH4', 'GENHLTH', 'EXRACT11', '_BMI5', 'DECIDE', '_ASTHMS1', '_MRACE1', '_LMTSCL1', 'ORNGDAY_', 
                     'MSCODE', 'PHYSHLTH', 'TOLDHI2', 'PADUR1_', 'DIFFWALK', 'BEANDAY_', 'CHCSCNCR', 'PAFREQ1_', '_AIDTST3', 
                     'FLSHTMY2', 'PA1VIGM_', 'METVL21_', 'SXORIENT', 'ARTHDIS2', 'PA1MIN_', 'MAXVO2_', '_RACE_G1', 'LONGWTCH', 
                     'SCNTWRK1', 'FEETCHK2', 'EMPLOY1', 'BLOODCHO', 'USENOW3', 'DRNKANY5', '_PAREC1', 'DIFFDRES', 'USEEQUIP', 
                     'ADDEPEV2', '_AGE80', 'HTM4', 'INCOME2', 'CVDSTRK3', 'CHCOCNCR', 'MEDCOST', 'PREDIAB1', 'BPMEDS', 'HAVARTH3', 
                     'CHECKUP1', 'TRNSGNDR', 'CHILDREN']

In [81]:
del x_indices["Id"]
x_data = clean_data(x_data, x_indices)

KeyError: 'Id'

In [ ]:
# keep only the columns of interest
x_data_interested = x_data[:, [x_indices[col] for col in final_feature_set]]
x_indices_interested = {col: i for i, col in enumerate(final_feature_set)}

# drop columns that are all Na
valid_cols = ~np.isnan(x_data_interested).all(axis=0)
x_data_interested = x_data_interested[:, valid_cols]
interesting_columns = [col for col, keep in zip(final_feature_set, valid_cols) if keep]
x_indices_interested = {col: i for i, col in enumerate(final_feature_set)}

# impute remaining NaNs with column mean
col_mean = np.nanmean(x_data_interested, axis=0)
inds = np.where(np.isnan(x_data_interested))
x_data_interested[inds] = np.take(col_mean, inds[1])

# standardize once
x_data_interested = (x_data_interested - np.mean(x_data_interested, axis=0)) / np.std(x_data_interested, axis=0)

# prepare y
y = y_data[:,1].copy()
y[np.isnan(y)] = np.nanmean(y)
y = (y + 1) / 2  # convert -1,1 → 0,1
y = y.flatten()

# checks
print("Unique labels:", np.unique(y))
print("Any NaN in features?", np.any(np.isnan(x_data_interested)))
print("Min std across features:", np.min(np.std(x_data_interested, axis=0)))
print("Label distribution (mean):", np.mean(y))

Unique labels: [0. 1.]
Any NaN in features? False
Min std across features: 0.9999999999999998
Label distribution (mean): 0.08830207079403295


# Evaluation

In [82]:
from utils.cv import (
    cross_validate,
    train_logreg,
    predict,
    compute_training_error,
)
from utils.sampling_methods import (
    undersample,
    oversample,
    smote
)


In [109]:
sampling_methods = {
    "none": None,
    # "undersample": undersample,
    # "oversample": oversample,
    # "smote": smote
}
sampling_kwargs = {
    # "undersample": {"ratio": 2.0, "seed": 42},
    # "oversample": {"ratio": 3.0, "seed": 42},
    # "smote": {"k": 5, "ratio": 3.0, "seed": 42, "batch_size": 20},
}

# all_results = []

In [112]:
all_results

[{'model': 'w_logistic',
  'sampling': 'none',
  0.75: {'error': np.float64(0.15043808188702953),
   'accuracy': np.float64(0.8495619181129704),
   'precision': np.float64(0.3153032913989803),
   'recall': np.float64(0.6005397511083608),
   'f1': np.float64(0.41348403405974166),
   'roc_auc': np.float64(0.14539892357989062)}},
 {'model': 'w_logistic',
  'sampling': 'undersample',
  0.75: {'error': np.float64(0.1502887531046686),
   'accuracy': np.float64(0.8497112468953313),
   'precision': np.float64(0.31561511410238163),
   'recall': np.float64(0.600623567441839),
   'f1': np.float64(0.413758493219994),
   'roc_auc': np.float64(0.14544039789551805)}},
 {'model': 'w_logistic',
  'sampling': 'oversample',
  0.75: {'error': np.float64(0.15040455909915257),
   'accuracy': np.float64(0.8495954409008473),
   'precision': np.float64(0.31520733156800845),
   'recall': np.float64(0.5997177461600518),
   'f1': np.float64(0.4132050208510857),
   'roc_auc': np.float64(0.14538644695916786)}},
 {'

In [ ]:
# CV for all logistic regression variants

for kind in ["w_logistic"]:
    print(f"\n=== MODEL: {kind.upper()} ===")

    for s_name, s_func in sampling_methods.items():
        print(f"\n--- Sampling: {s_name.upper()} ---")

        # cross-validation
        results = cross_validate(
            model_fn=lambda y, X: train_logreg(
                y, X,
                kind=kind,
                gamma=1,
                lambda_ = 0.01,
                max_iters=500,
                verbose=True
            ),
            X=x_data_interested,
            y=y,
            k_folds=5,
            seed=42,
            sampling_fn=s_func,
            sampling_kwargs=sampling_kwargs.get(s_name, None),
            predict_fn=lambda X, w: (predict(X, w), predict(X, w, return_prob=True)),
            metric_fn=compute_training_error,
            thresholds = [0.5, 0.7, 0.75, 0.8, 0.85],
            use_predictions=False,
            verbose=True
        )

        # store results
        model_result = {"model": kind, "sampling": s_name, **results}
        all_results.append(model_result)
        print("Average CV metrics:", results)



=== MODEL: W_LOGISTIC ===

--- Sampling: NONE ---
[w_logistic] Training finished — final loss: 0.518717
Fold 1/5 @ thr=0.50 → error:0.385, accuracy:0.615, precision:0.176, recall:0.922, f1:0.295, roc_auc:0.144
Fold 1/5 @ thr=0.70 → error:0.192, accuracy:0.808, precision:0.272, recall:0.716, f1:0.395, roc_auc:0.144
Fold 1/5 @ thr=0.75 → error:0.155, accuracy:0.845, precision:0.309, recall:0.629, f1:0.414, roc_auc:0.144
Fold 1/5 @ thr=0.80 → error:0.124, accuracy:0.876, precision:0.356, recall:0.518, f1:0.422, roc_auc:0.144
Fold 1/5 @ thr=0.85 → error:0.102, accuracy:0.898, precision:0.410, recall:0.382, f1:0.395, roc_auc:0.144
[w_logistic] Training finished — final loss: 0.518503
Fold 2/5 @ thr=0.50 → error:0.383, accuracy:0.617, precision:0.179, recall:0.925, f1:0.300, roc_auc:0.145
Fold 2/5 @ thr=0.70 → error:0.189, accuracy:0.811, precision:0.279, recall:0.710, f1:0.400, roc_auc:0.145
Fold 2/5 @ thr=0.75 → error:0.153, accuracy:0.847, precision:0.314, recall:0.615, f1:0.416, roc_auc

In [99]:
output_dir = os.path.join(os.getcwd(), '..', 'results', 'eval')
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'all_results.json')
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=4, ensure_ascii=False)

In [ ]:
# grid search values
gammas = [1.5,1.7,2]
lambdas = [0.01]
max_iters_list = [500]

# to store results
# grid_results = []

for gamma in gammas:
    for lambda_ in lambdas:
        for max_iters in max_iters_list : 
            print(f"\n=== Weighted Logistic | γ={gamma} | λ={lambda_} | max_iters={max_iters} ===")

            for s_name, s_func in sampling_methods.items():
                print(f"\n--- Sampling: {s_name.upper()} ---")

                results = cross_validate(
                    model_fn=lambda y, X: train_logreg(
                        y, X,
                        kind="w_logistic",
                        gamma=gamma,
                        lambda_=lambda_,
                        max_iters=max_iters,
                        tol=1e-4,
                        verbose=False
                    ),
                    X=x_data_interested,
                    y=y,
                    k_folds=5,
                    seed=42,
                    sampling_fn=s_func,
                    sampling_kwargs=sampling_kwargs.get(s_name, None),
                    predict_fn=lambda X, w: (predict(X, w), predict(X, w, return_prob=True)),
                    metric_fn=compute_training_error,
                    thresholds=0.75,       # keep your fixed threshold
                    use_predictions=False,
                    verbose=True
                )

                # store results
                combo_result = {
                    "model": "w_logistic",
                    "sampling": s_name,
                    "gamma": gamma,
                    "lambda": lambda_,
                    "max_iters" : max_iters,
                    **results
                }
                grid_results.append(combo_result)
                print("Average CV metrics:", results)

# save all grid search results
output_dir = os.path.join(os.getcwd(), '..', 'results', 'eval')
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'gridsearch_wlogreg.json')

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(grid_results, f, indent=4, ensure_ascii=False)

print(f"\nGrid search complete. Results saved to {output_path}")


=== Weighted Logistic | γ=1.5 | λ=0.01 | max_iters=500 ===

--- Sampling: NONE ---
Fold 1/5 @ thr=0.75 → error:0.181, accuracy:0.819, precision:0.278, recall:0.673, f1:0.393, roc_auc:0.151
Fold 2/5 @ thr=0.75 → error:0.181, accuracy:0.819, precision:0.280, recall:0.662, f1:0.393, roc_auc:0.152
Fold 3/5 @ thr=0.75 → error:0.180, accuracy:0.820, precision:0.284, recall:0.674, f1:0.400, roc_auc:0.147
Fold 4/5 @ thr=0.75 → error:0.184, accuracy:0.816, precision:0.277, recall:0.677, f1:0.393, roc_auc:0.147
Fold 5/5 @ thr=0.75 → error:0.184, accuracy:0.816, precision:0.277, recall:0.669, f1:0.392, roc_auc:0.149
Average CV metrics: {0.75: {'error': np.float64(0.1821628293232981), 'accuracy': np.float64(0.8178371706767018), 'precision': np.float64(0.27903614135093163), 'recall': np.float64(0.6710769677095434), 'f1': np.float64(0.3941608886120176), 'roc_auc': np.float64(0.149266454436784)}}

=== Weighted Logistic | γ=1.7 | λ=0.01 | max_iters=500 ===

--- Sampling: NONE ---
Fold 1/5 @ thr=0.75 

KeyboardInterrupt: 